# RivalRadar Agentic Pipeline (Agent 1-4)

Single notebook pipeline:
- Agent 1: scrape + parse competitor pages
- Agent 2: Groq (llama-3.3-70b-versatile) portfolio risk analysis
- Agent 3: Groq impact forecasting
- Agent 4: Groq VC action recommendations

Goal: run one end-to-end pipeline in one notebook.

## 1. Imports, API Keys, and Shared Configuration

In [9]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "groq", "--quiet"], check=True)

import asyncio
import json
import os
import re
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from openai import OpenAI

from competitor_targets import COMPETITOR_TARGETS
from scrapers.output_parser import parse_pricing_page, save_structured

# Crawlee is optional for live scraping; fallback mode can still run without it.
try:
    from scrapers.crawlee_scraper import CrawleeScraper
    CRAWLEE_AVAILABLE = True
except ModuleNotFoundError:
    CrawleeScraper = None
    CRAWLEE_AVAILABLE = False

OUTPUT_DIR = Path("scraper_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── API Keys ──────────────────────────────────────────────────────────────────
# Load from .env.example in the same directory as this notebook.
def _load_env_file(path: Path) -> dict:
    env = {}
    if path.exists():
        for line in path.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, _, v = line.partition("=")
                env[k.strip()] = v.strip().strip("'\"")
    return env

_env = _load_env_file(Path(".env.example"))
OPENAI_API_KEY = _env.get("OPENAI_API_KEY", "").strip()
GROQ_API_KEY = _env.get("GROQ_API_KEY", "").strip()

if not GROQ_API_KEY and not OPENAI_API_KEY:
    raise ValueError("No API key found. Add GROQ_API_KEY or OPENAI_API_KEY to .env.example")
# ─────────────────────────────────────────────────────────────────────────────

if GROQ_API_KEY:
    OPENAI_MODEL = "llama-3.3-70b-versatile"
else:
    OPENAI_MODEL = _env.get("OPENAI_MODEL", "gpt-4o-mini")

# ── Portfolio Company Context ─────────────────────────────────────────────────
# Describes the VC-backed portfolio company being protected. Agents 2, 3, and 4
# use this to frame risk scores and recommendations relative to this company.
# Edit these fields to match your actual portfolio company.
PORTFOLIO_CONTEXT = {
    "name": "RivalRadar Portfolio Co",
    "description": (
        "A VC-backed B2B SaaS company offering workflow automation and project "
        "intelligence tooling for mid-market teams (50-500 seats). Competes primarily "
        "with productivity, CRM, and project management platforms."
    ),
    "stage": "Series B",
    "arr_range": "$5M-$15M",
    "primary_segments": ["mid-market", "SMB"],
    "key_value_props": [
        "AI-native workflow automation",
        "cross-tool analytics and reporting",
        "lightweight CRM integration layer",
    ],
    "at_risk_from": ["pricing pressure", "feature parity", "bundling by larger platforms"],
}
# ─────────────────────────────────────────────────────────────────────────────

RUN_LIVE_SCRAPE = CRAWLEE_AVAILABLE
VERBOSE = True
MAX_RETRIES = 1

print(f"Groq key loaded: {bool(GROQ_API_KEY)}")
print(f"OpenAI key loaded: {bool(OPENAI_API_KEY)}")
print(f"Active model: {OPENAI_MODEL}")
print(f"Output dir: {OUTPUT_DIR.resolve()}")
print(f"Competitors configured: {len(COMPETITOR_TARGETS)}")
print(f"Crawlee available: {CRAWLEE_AVAILABLE}")
print(f"Live scrape mode: {RUN_LIVE_SCRAPE}")

if not CRAWLEE_AVAILABLE:
    print("Install missing deps in your venv: pip install -r requirements.txt")


def get_openai_client() -> OpenAI:
    if GROQ_API_KEY:
        print("Using Groq inference (llama-3.3-70b-versatile)")
        return OpenAI(
            base_url="https://api.groq.com/openai/v1",
            api_key=GROQ_API_KEY,
        )
    elif OPENAI_API_KEY:
        print("Using OpenAI inference")
        return OpenAI(api_key=OPENAI_API_KEY)
    else:
        raise ValueError("No API key found. Add GROQ_API_KEY or OPENAI_API_KEY to .env.example")


def chat_json(client: OpenAI, system_prompt: str, user_prompt: str, temperature: float = 0.2) -> dict[str, Any]:
    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        response_format={"type": "json_object"},
    )
    content = response.choices[0].message.content
    if not content:
        raise RuntimeError("LLM returned empty content")
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        cleaned = re.sub(r"```json|```", "", content).strip()
        return json.loads(cleaned)


def coerce_float(v, default=0.0):
    return float(v) if v is not None else float(default)


def coerce_int(v, default=0):
    return int(v) if v is not None else int(default)


Groq key loaded: True
OpenAI key loaded: False
Active model: llama-3.3-70b-versatile
Output dir: /Users/pranay/Documents/RivalRadar/DBA5115-rivalradar/rivalradar/scraper_output
Competitors configured: 5
Crawlee available: True
Live scrape mode: True


## 2. Define First Agent Interface and Implementation

In [10]:
def assess_data_sufficiency(profile: dict) -> dict:
    """
    Returns a sufficiency assessment for a structured profile.
    A profile is considered insufficient if the scrape yielded too little
    content to support meaningful downstream analysis.
    """
    raw_chars = profile.get("raw_char_count", 0)
    plans = profile.get("plans", [])
    pricing_summary = profile.get("pricing_summary", {})

    insufficient = (
        raw_chars < 300
        or len(plans) == 0
        or pricing_summary.get("price_range", "N/A") == "N/A"
    )

    return {
        "sufficient": not insufficient,
        "raw_char_count": raw_chars,
        "plan_count": len(plans),
        "reason": (
            "Insufficient: scrape returned too little content for analysis."
            if insufficient
            else "Sufficient: profile contains pricing structure and plan data."
        ),
    }


class Agent1Collector:
    """Collect competitor pricing pages and emit structured competitor profiles."""

    def __init__(self, output_dir: Path):
        self.output_dir = output_dir
        self.scraper = CrawleeScraper() if CRAWLEE_AVAILABLE else None

    def build_targets(self) -> list[dict[str, str]]:
        return [
            {
                "url": target["pricing_url"],
                "page_type": "pricing",
                "name": target["name"],
            }
            for target in COMPETITOR_TARGETS
        ]

    async def collect(self) -> dict[str, Any]:
        if self.scraper is None:
            raise ModuleNotFoundError(
                "crawlee is not installed. Install deps with: pip install -r requirements.txt"
            )

        targets = self.build_targets()
        scrape_results = await self.scraper.scrape_pages(targets)

        profiles: list[dict[str, Any]] = []
        for result in scrape_results:
            profile = parse_pricing_page(
                raw_text=result.get("raw_text", ""),
                company=result.get("name", "Unknown"),
                url=result.get("url", ""),
                scraped_at=result.get("scraped_at", ""),
            )
            profile["data_sufficiency"] = assess_data_sufficiency(profile)
            save_structured(profile, self.output_dir)
            profiles.append(profile)

        return {
            "agent": "agent1_collector",
            "target_count": len(targets),
            "scrape_result_count": len(scrape_results),
            "structured_profiles": profiles,
        }


def load_existing_structured_profiles(output_dir: Path) -> list[dict[str, Any]]:
    profiles: list[dict[str, Any]] = []
    for path in sorted(output_dir.glob("*_structured.json")):
        with open(path, encoding="utf-8") as f:
            profile = json.load(f)
        if "data_sufficiency" not in profile:
            profile["data_sufficiency"] = assess_data_sufficiency(profile)
        profiles.append(profile)
    return profiles


## 2b. Synthetic Event Generator (fallback for insufficient Agent 1 data)

In [11]:
EVENT_TYPES = ["PRICE_CUT", "FEATURE_LAUNCH", "INTEGRATION_ANNOUNCEMENT", "TIER_RESTRUCTURE"]


@dataclass
class SyntheticCompetitorEvent:
    company: str
    event_type: str
    event_description: str
    simulated_detail: dict[str, Any]
    is_synthetic: bool
    generated_at: str


class SyntheticEventGenerator:
    """
    When Agent 1 returns a profile with insufficient data,
    this generator produces a plausible but clearly labelled synthetic
    competitive event to drive downstream pipeline stages.
    """

    def __init__(self, client: OpenAI):
        self.client = client

    def generate(self, profile: dict[str, Any]) -> SyntheticCompetitorEvent:
        company = profile.get("company", "Unknown")
        partial_context = {
            "company": company,
            "available_data": profile,
            "note": (
                "The scraped data for this company was sparse. "
                "Generate a realistic but plausible synthetic competitive event "
                "that a company like this might plausibly announce. "
                "Base the event on publicly known characteristics of this company "
                "if available in your training knowledge."
            ),
        }

        system_prompt = (
            "You are a competitive intelligence scenario designer for a venture capital firm. "
            "Your job is to generate realistic synthetic competitive events for demo and simulation purposes. "
            "These events are clearly labelled as synthetic and are used to test how a VC portfolio "
            "risk monitoring system would respond to different market moves. "
            "Return ONLY valid JSON. No markdown, no explanation text.\n\n"
            "The JSON must contain exactly these keys:\n"
            "- event_type: one of PRICE_CUT, FEATURE_LAUNCH, INTEGRATION_ANNOUNCEMENT, TIER_RESTRUCTURE\n"
            "- event_description: string (1-2 sentences describing the event as if it just happened)\n"
            "- simulated_detail: object with relevant fields depending on event_type:\n"
            "  For PRICE_CUT: {affected_plan, old_price, new_price, discount_pct}\n"
            "  For FEATURE_LAUNCH: {feature_name, target_segment, replaces_integration}\n"
            "  For INTEGRATION_ANNOUNCEMENT: {partner, integration_type, affected_workflows}\n"
            "  For TIER_RESTRUCTURE: {change_description, affected_tiers, seat_cap_change}\n\n"
            "Make the event specific, realistic, and commercially plausible. "
            "Do not invent fictional company names. Use the actual company name provided."
        )

        user_prompt = (
            f"Generate a synthetic competitive event for {company}.\n"
            f"Context from partial scrape: {json.dumps(partial_context, ensure_ascii=False)}\n\n"
            "Pick the event type that would be most threatening to a VC-backed SaaS portfolio company "
            "competing in the same market segment. Be specific about pricing figures or feature names."
        )

        payload = chat_json(self.client, system_prompt, user_prompt, temperature=0.4)

        event_type = str(payload.get("event_type", "PRICE_CUT")).upper().strip()
        if event_type not in EVENT_TYPES:
            event_type = "PRICE_CUT"

        return SyntheticCompetitorEvent(
            company=company,
            event_type=event_type,
            event_description=str(payload.get("event_description", "Synthetic event generated.")).strip(),
            simulated_detail=payload.get("simulated_detail", {}),
            is_synthetic=True,
            generated_at=datetime.now(timezone.utc).isoformat(),
        )


## 3. Define Agent 2 (OpenAI Vulnerability Analysis)

In [12]:
@dataclass
class ScoreComponent:
    name: str
    weight: float
    score: float
    weighted_score: float
    reasoning: str


@dataclass
class VulnerabilityResult:
    company: str
    portfolio_risk_score: float
    risk_level: str
    confidence: float
    reasoning_summary: str
    detailed_reasoning: list[str]
    decision_trace: list[str]
    component_breakdown: list[ScoreComponent]
    signals: dict[str, int]
    metrics: dict[str, Any]
    scraped_at: str
    peer_rank: int | None
    peer_percentile: float | None


class Agent2CompetitiveAnalyzer:
    """Analyze competitor profiles with OpenAI and return explainable vulnerability results."""

    component_weights = {
        "pricing_pressure": 0.28,
        "segment_coverage": 0.22,
        "feature_depth": 0.18,
        "strategic_signals": 0.20,
        "business_model_pressure": 0.12,
    }

    def __init__(self, output_dir: Path, client: OpenAI):
        self.output_dir = output_dir
        self.client = client

    def _analyze_one(self, profile: dict[str, Any]) -> VulnerabilityResult:
        system_prompt = (
            "You are a portfolio risk analyst at a venture capital firm. You assess how competitive events "
            "and market moves by rival companies threaten the market position, revenue, and exit value of "
            "VC-backed SaaS portfolio companies.\n\n"
            "Return ONLY valid JSON. No markdown, no explanation text, no code fences.\n\n"
            "Required top-level keys:\n"
            "- portfolio_risk_score: float 0.0 to 1.0 (how much does this competitor threaten the portfolio company's position)\n"
            "- confidence: float 0.0 to 1.0\n"
            "- risk_level: one of \"low\", \"medium\", \"high\", \"critical\"\n"
            "- reasoning_summary: string (2-3 sentences, framed around portfolio company impact)\n"
            "- detailed_reasoning: list of strings (one per scoring dimension)\n"
            "- decision_trace: list of strings (step-by-step logic)\n"
            "- component_breakdown: list of exactly 5 objects with keys: name, score (float 0-1), reasoning\n"
            "- signals: object with keys ai_momentum, enterprise_readiness, integration_strength (each 0-10)\n"
            "- metrics: object with keys plan_count, price_range, raw_char_count, avg_features_per_plan, "
            "unique_feature_count, numeric_price_count\n\n"
            "component_breakdown must use exactly these names:\n"
            "1. pricing_pressure\n"
            "2. segment_coverage\n"
            "3. feature_depth\n"
            "4. strategic_signals\n"
            "5. business_model_pressure\n\n"
            "Scoring rubric:\n"
            "- 0.0-0.25: Minimal threat to portfolio company position\n"
            "- 0.25-0.50: Early-stage threat, monitor\n"
            "- 0.50-0.75: Active threat, portco leadership should be aware\n"
            "- 0.75-1.0: Severe near-term threat to revenue or customer retention\n\n"
            "Score honestly and independently. Most companies should score between 0.3 and 0.7. "
            "A score of 1.0 requires explicit justification from the data."
        )
        user_prompt = (
            "Assess the portfolio risk this competitor poses to a VC-backed SaaS portfolio company "
            "operating in the same market segment.\n\n"
            "If a synthetic_event field is present in the profile data, treat it as a real recent competitive "
            "move and weight your analysis around its likely impact on customer churn, win/loss rates, and "
            "pricing power at the portfolio company.\n\n"
            "If the profile data is sparse, use the synthetic_event as the primary basis for your assessment. "
            "Do not penalise the score for data sparsity — score based on what the event implies.\n\n"
            f"Portfolio company context: {json.dumps(PORTFOLIO_CONTEXT, ensure_ascii=False)}\n\n"
            f"Competitor profile: {json.dumps(profile, ensure_ascii=False)}"
        )
        payload = chat_json(
            self.client,
            system_prompt,
            user_prompt,
            temperature=0.1,
        )

        raw_component_names = [
            item.get("name", "")
            for item in payload.get("component_breakdown", [])
            if isinstance(item, dict)
        ]
        if VERBOSE:
            print(f"[Agent2] Raw component names from LLM: {raw_component_names}")

        components_by_name = {
            item.get("name", "").lower().strip().replace(" ", "_"): item
            for item in payload.get("component_breakdown", [])
            if isinstance(item, dict)
        }

        components: list[ScoreComponent] = []
        for name, weight in self.component_weights.items():
            raw = components_by_name.get(name, {})
            score = max(0.0, min(1.0, coerce_float(raw.get("score"), 0.0)))
            components.append(
                ScoreComponent(
                    name=name,
                    weight=weight,
                    score=round(score, 3),
                    weighted_score=round(score * weight, 4),
                    reasoning=str(raw.get("reasoning") or "No reasoning provided.").strip(),
                )
            )

        portfolio_risk_score = max(0.0, min(1.0, coerce_float(payload.get("portfolio_risk_score"), sum(c.weighted_score for c in components))))
        confidence = max(0.0, min(1.0, coerce_float(payload.get("confidence"), 0.6)))
        risk_level = str(payload.get("risk_level") or "medium").lower().strip()
        if risk_level not in {"low", "medium", "high", "critical"}:
            risk_level = "high" if portfolio_risk_score >= 0.65 else ("medium" if portfolio_risk_score >= 0.45 else "low")

        signals_raw = payload.get("signals", {}) if isinstance(payload.get("signals"), dict) else {}
        signals = {
            "ai_momentum": coerce_int(signals_raw.get("ai_momentum"), 0),
            "enterprise_readiness": coerce_int(signals_raw.get("enterprise_readiness"), 0),
            "integration_strength": coerce_int(signals_raw.get("integration_strength"), 0),
        }

        metrics_raw = payload.get("metrics", {}) if isinstance(payload.get("metrics"), dict) else {}
        metrics = {
            "plan_count": coerce_int(metrics_raw.get("plan_count"), len(profile.get("plans") or [])),
            "price_range": metrics_raw.get("price_range") or (profile.get("pricing_summary") or {}).get("price_range", "N/A"),
            "raw_char_count": coerce_int(metrics_raw.get("raw_char_count"), profile.get("raw_char_count", 0)),
            "avg_features_per_plan": coerce_float(metrics_raw.get("avg_features_per_plan"), 0.0),
            "unique_feature_count": coerce_int(metrics_raw.get("unique_feature_count"), 0),
            "numeric_price_count": coerce_int(metrics_raw.get("numeric_price_count"), 0),
        }

        detailed_reasoning = payload.get("detailed_reasoning", [])
        if not isinstance(detailed_reasoning, list):
            detailed_reasoning = [str(detailed_reasoning)]

        decision_trace = payload.get("decision_trace", [])
        if not isinstance(decision_trace, list):
            decision_trace = [str(decision_trace)]

        return VulnerabilityResult(
            company=profile.get("company", "Unknown"),
            portfolio_risk_score=round(portfolio_risk_score, 3),
            risk_level=risk_level,
            confidence=round(confidence, 3),
            reasoning_summary=str(payload.get("reasoning_summary", "No summary provided.")).strip(),
            detailed_reasoning=[str(x) for x in detailed_reasoning],
            decision_trace=[str(x) for x in decision_trace],
            component_breakdown=components,
            signals=signals,
            metrics=metrics,
            scraped_at=profile.get("scraped_at", datetime.now(timezone.utc).isoformat()),
            peer_rank=None,
            peer_percentile=None,
        )

    def analyze(self, agent1_output: dict[str, Any]) -> dict[str, Any]:
        profiles = agent1_output["structured_profiles"]
        results = []
        for profile in profiles:
            results.append(self._analyze_one(profile))
            time.sleep(1)
        results.sort(key=lambda r: r.portfolio_risk_score, reverse=True)

        total = len(results)
        for idx, result in enumerate(results, start=1):
            result.peer_rank = idx
            result.peer_percentile = 1.0 if total == 1 else round(1 - ((idx - 1) / (total - 1)), 3)
            result.reasoning_summary = (
                f"Ranked #{result.peer_rank}/{total} (peer_percentile={result.peer_percentile:.3f}). "
                f"{result.reasoning_summary}"
            )

        report_path = self.output_dir / "portfolio_risk_report.json"
        with open(report_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "generated_at": datetime.now(timezone.utc).isoformat(),
                    "model": f"groq_{OPENAI_MODEL}_portfolio_risk_v1",
                    "results": [asdict(r) for r in results],
                },
                f,
                indent=2,
                ensure_ascii=False,
            )

        return {
            "agent": "agent2_analyzer",
            "vulnerability_results": results,
            "report_path": str(report_path),
            "analyzed_count": len(results),
        }

## 4. Define Agent 3 (Pricing Predictor)

In [13]:
@dataclass
class ImpactForecast:
    company: str
    impact_probability: float
    predicted_timeline: str
    risk_level: str
    reasoning_summary: str
    drivers: dict[str, float]
    decision_trace: list[str]
    evidence: dict[str, Any]
    generated_at: str


class Agent3ImpactForecaster:
    """Forecast competitive impact on portfolio company metrics using OpenAI."""

    def __init__(self, output_dir: Path, client: OpenAI):
        self.output_dir = output_dir
        self.client = client

    def _predict_one(self, vuln: VulnerabilityResult, profile: dict[str, Any]) -> ImpactForecast:
        context = {
            "company": vuln.company,
            "portfolio_risk_score": vuln.portfolio_risk_score,
            "risk_level": vuln.risk_level,
            "confidence": vuln.confidence,
            "signals": vuln.signals,
            "metrics": vuln.metrics,
            "profile": profile,
        }
        system_prompt = (
            "You are an investment risk forecaster at a venture capital firm. You estimate how quickly "
            "a competitive event will erode a portfolio company's revenue, retention, or pricing power.\n\n"
            "Return ONLY valid JSON. No markdown.\n\n"
            "Required keys:\n"
            "- impact_probability: float 0.0-1.0 (likelihood this event materially affects portfolio company in the next 6 months)\n"
            "- predicted_timeline: one of \"0-30 days\", \"1-2 months\", \"2-3 months\", \"3+ months\"\n"
            "- risk_level: one of \"low\", \"medium\", \"high\"\n"
            "- reasoning_summary: 2-3 sentences framed around investment risk\n"
            "- drivers: object with keys vulnerability_pressure, strategic_signal_intensity, plan_complexity, "
            "portfolio_competition_intensity, pricing_plan_volatility, analysis_confidence (each float 0-1)\n"
            "- decision_trace: list of strings\n"
            "- evidence: object with supporting data points"
        )
        user_prompt = (
            "Forecast how quickly this competitive event or market move could affect the portfolio company's metrics.\n\n"
            "Focus specifically on: customer churn risk, win/loss rate degradation, and pricing power compression.\n\n"
            "If the competitor profile contains a synthetic_event, base your timeline on the type of event:\n"
            "- PRICE_CUT: fastest impact, often 0-30 days on renewal cycles\n"
            "- FEATURE_LAUNCH: medium, 1-3 months depending on customer awareness\n"
            "- INTEGRATION_ANNOUNCEMENT: slower, 2-4 months\n"
            "- TIER_RESTRUCTURE: variable\n\n"
            f"Portfolio company: {json.dumps(PORTFOLIO_CONTEXT, ensure_ascii=False)}\n"
            f"Competitor: {json.dumps(context, ensure_ascii=False)}"
        )

        payload = chat_json(self.client, system_prompt, user_prompt, temperature=0.15)

        probability = max(0.0, min(1.0, coerce_float(payload.get("impact_probability"), 0.0)))
        risk = str(payload.get("risk_level") or "low").lower().strip()
        if risk not in {"low", "medium", "high"}:
            risk = "high" if probability >= 0.75 else ("medium" if probability >= 0.5 else "low")

        timeline = str(payload.get("predicted_timeline") or "3+ months").strip()
        if timeline not in {"0-30 days", "1-2 months", "2-3 months", "3+ months"}:
            timeline = "0-30 days" if probability >= 0.8 else ("1-2 months" if probability >= 0.65 else ("2-3 months" if probability >= 0.5 else "3+ months"))

        raw_drivers = payload.get("drivers", {}) if isinstance(payload.get("drivers"), dict) else {}
        drivers = {
            "vulnerability_pressure": round(max(0.0, min(1.0, coerce_float(raw_drivers.get("vulnerability_pressure"), vuln.portfolio_risk_score))), 3),
            "strategic_signal_intensity": round(max(0.0, min(1.0, coerce_float(raw_drivers.get("strategic_signal_intensity"), 0.0))), 3),
            "plan_complexity": round(max(0.0, min(1.0, coerce_float(raw_drivers.get("plan_complexity"), 0.0))), 3),
            "portfolio_competition_intensity": round(max(0.0, min(1.0, coerce_float(raw_drivers.get("portfolio_competition_intensity"), 0.0))), 3),
            "pricing_plan_volatility": round(max(0.0, min(1.0, coerce_float(raw_drivers.get("pricing_plan_volatility"), 0.0))), 3),
            "analysis_confidence": round(max(0.0, min(1.0, coerce_float(raw_drivers.get("analysis_confidence"), vuln.confidence))), 3),
        }

        decision_trace = payload.get("decision_trace", [])
        if not isinstance(decision_trace, list):
            decision_trace = [str(decision_trace)]

        evidence = payload.get("evidence", {}) if isinstance(payload.get("evidence"), dict) else {}
        evidence.setdefault("price_range", vuln.metrics.get("price_range", "N/A"))
        evidence.setdefault("peer_rank", vuln.peer_rank)

        return ImpactForecast(
            company=vuln.company,
            impact_probability=round(probability, 3),
            predicted_timeline=timeline,
            risk_level=risk,
            reasoning_summary=str(payload.get("reasoning_summary", "No summary provided.")).strip(),
            drivers=drivers,
            decision_trace=[str(x) for x in decision_trace],
            evidence=evidence,
            generated_at=datetime.now(timezone.utc).isoformat(),
        )

    def predict(self, agent1_output: dict[str, Any], agent2_output: dict[str, Any]) -> dict[str, Any]:
        profiles = {p.get("company", "Unknown"): p for p in agent1_output["structured_profiles"]}
        vulnerability_results = agent2_output["vulnerability_results"]

        predictions = []
        for vuln in vulnerability_results:
            predictions.append(self._predict_one(vuln, profiles.get(vuln.company, {})))
            time.sleep(1)
        predictions.sort(key=lambda p: p.impact_probability, reverse=True)

        report_path = self.output_dir / "impact_forecast_report.json"
        with open(report_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "generated_at": datetime.now(timezone.utc).isoformat(),
                    "model": f"groq_{OPENAI_MODEL}_impact_forecast_v1",
                    "results": [asdict(p) for p in predictions],
                },
                f,
                indent=2,
                ensure_ascii=False,
            )

        return {
            "agent": "agent3_pricing",
            "impact_forecasts": predictions,
            "report_path": str(report_path),
            "forecast_count": len(predictions),
        }

## 5. Define Agent 4 (Action Planner)

In [14]:
@dataclass
class ActionRecommendation:
    company: str
    priority: str
    recommendation_type: str
    owner: str
    due_window: str
    action_title: str
    action_detail: str
    rationale: str
    evidence: dict[str, Any]
    decision_trace: list[str]
    generated_at: str


class Agent4ActionPlanner:
    """Generate VC-facing action recommendations from Agent 2 + Agent 3 with OpenAI."""

    def __init__(self, output_dir: Path, client: OpenAI):
        self.output_dir = output_dir
        self.client = client

    def _recommend_one(self, vuln: VulnerabilityResult, pred: ImpactForecast) -> ActionRecommendation:
        context = {
            "company": vuln.company,
            "portfolio_company": PORTFOLIO_CONTEXT,
            "portfolio_risk_score": vuln.portfolio_risk_score,
            "vulnerability_risk": vuln.risk_level,
            "vulnerability_summary": vuln.reasoning_summary,
            "top_components": [
                {"name": c.name, "score": c.score, "reasoning": c.reasoning}
                for c in sorted(vuln.component_breakdown, key=lambda x: x.weighted_score, reverse=True)[:3]
            ],
            "impact_probability": pred.impact_probability,
            "impact_timeline": pred.predicted_timeline,
            "impact_risk": pred.risk_level,
            "impact_summary": pred.reasoning_summary,
            "top_impact_drivers": sorted(pred.drivers.items(), key=lambda x: x[1], reverse=True)[:3],
        }
        system_prompt = (
            "You are a portfolio operations lead at a venture capital firm. You generate specific, "
            "actionable recommendations for the investment team — not for the portfolio company itself.\n\n"
            "Your recommendations cover what the VC can do: board-level conversations, follow-on positioning, "
            "risk flagging, or proactive portco engagement. You do not tell the portfolio company to change its "
            "product or pricing — that is outside the VC's direct control.\n\n"
            "Return ONLY valid JSON. No markdown.\n\n"
            "Required keys:\n"
            "- priority: one of P0, P1, P2, P3\n"
            "  P0: immediate board flag, revenue impact likely within 30 days\n"
            "  P1: portfolio ops briefing required within 2 weeks\n"
            "  P2: add to next quarterly portfolio review\n"
            "  P3: log and monitor, no immediate action\n"
            "- recommendation_type: one of pricing_response, product_response, gtm_response, monitor_only\n"
            "- owner: who at the VC firm takes point (e.g. \"Portfolio Operations Lead\", \"Lead Partner\", \"Board Observer\")\n"
            "- due_window: specific timeframe\n"
            "- action_title: 5-8 words\n"
            "- action_detail: 2-3 sentences of specific action steps the VC team takes\n"
            "- rationale: how this protects investment value\n"
            "- evidence: supporting data\n"
            "- decision_trace: list of strings"
        )
        user_prompt = (
            "Generate one actionable recommendation for the VC investment team.\n\n"
            "This is NOT advice for the portfolio company. This is what the VC firm's portfolio operations "
            "team should do to protect the value of their investment.\n\n"
            "Assign P0 only if impact_probability > 0.80 AND portfolio_risk_score > 0.75.\n"
            "Assign P1 if either threshold is met.\n"
            "Assign P2 for active monitoring posture.\n"
            "Assign P3 for log-and-watch.\n\n"
            "If the competitive event is synthetic (is_synthetic: true in the profile), note this in the "
            "action_detail but still treat the scenario as worth planning for — the value of early warning "
            "is preparation, not just reaction.\n\n"
            f"Context: {json.dumps(context, ensure_ascii=False)}"
        )

        payload = chat_json(self.client, system_prompt, user_prompt, temperature=0.2)
        priority = str(payload.get("priority", "P2")).upper().strip()
        if priority not in {"P0", "P1", "P2", "P3"}:
            priority = "P2"

        rec_type = str(payload.get("recommendation_type", "monitor_only")).lower().strip()
        if rec_type not in {"pricing_response", "product_response", "gtm_response", "monitor_only"}:
            rec_type = "monitor_only"

        decision_trace = payload.get("decision_trace", [])
        if not isinstance(decision_trace, list):
            decision_trace = [str(decision_trace)]

        evidence = payload.get("evidence", {}) if isinstance(payload.get("evidence"), dict) else {}
        evidence.setdefault("portfolio_risk_score", vuln.portfolio_risk_score)
        evidence.setdefault("impact_probability", pred.impact_probability)

        return ActionRecommendation(
            company=vuln.company,
            priority=priority,
            recommendation_type=rec_type,
            owner=str(payload.get("owner", "Portfolio Operations Lead")).strip(),
            due_window=str(payload.get("due_window", "14 days")).strip(),
            action_title=str(payload.get("action_title", f"Monitor {vuln.company}")).strip(),
            action_detail=str(payload.get("action_detail", "No detail provided.")).strip(),
            rationale=str(payload.get("rationale", "No rationale provided.")).strip(),
            evidence=evidence,
            decision_trace=[str(x) for x in decision_trace],
            generated_at=datetime.now(timezone.utc).isoformat(),
        )

    def recommend(self, agent2_output: dict[str, Any], agent3_output: dict[str, Any]) -> dict[str, Any]:
        vulnerability_results = agent2_output["vulnerability_results"]
        predictions = {p.company: p for p in agent3_output["impact_forecasts"]}

        recommendations: list[ActionRecommendation] = []
        for vuln in vulnerability_results:
            pred = predictions.get(vuln.company)
            if pred is None:
                continue
            recommendations.append(self._recommend_one(vuln, pred))

        priority_order = {"P0": 0, "P1": 1, "P2": 2, "P3": 3}
        recommendations.sort(key=lambda r: (priority_order.get(r.priority, 9), r.company))

        report_path = self.output_dir / "action_recommendations_report.json"
        with open(report_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "generated_at": datetime.now(timezone.utc).isoformat(),
                    "model": f"groq_{OPENAI_MODEL}_action_recommendation_v1",
                    "results": [asdict(r) for r in recommendations],
                },
                f,
                indent=2,
                ensure_ascii=False,
            )

        return {
            "agent": "agent4_actions",
            "action_recommendations": recommendations,
            "report_path": str(report_path),
            "recommendation_count": len(recommendations),
        }

## 6. Compose Multi-Agent Pipeline (Agents 1-4)

In [15]:
async def run_agent_pipeline(run_live_scrape: bool = True) -> dict[str, Any]:
    start = time.perf_counter()

    client = get_openai_client()

    agent1 = Agent1Collector(OUTPUT_DIR)
    agent2 = Agent2CompetitiveAnalyzer(OUTPUT_DIR, client)
    agent3 = Agent3ImpactForecaster(OUTPUT_DIR, client)
    agent4 = Agent4ActionPlanner(OUTPUT_DIR, client)

    agent1_output: dict[str, Any] | None = None

    if run_live_scrape:
        for attempt in range(MAX_RETRIES + 1):
            try:
                if VERBOSE:
                    print(f"[Agent1] Live scrape attempt {attempt + 1}/{MAX_RETRIES + 1}")
                agent1_output = await agent1.collect()
                if VERBOSE:
                    print(f"[Agent1] Scraped {len(agent1_output.get('structured_profiles', []))} profiles.")
                break
            except Exception as exc:
                print(f"[Agent1] Attempt failed: {exc}")
                if attempt == MAX_RETRIES:
                    print("[Agent1] Falling back to existing structured JSON files.")

    # Fall back if scrape was skipped, failed, or returned no profiles
    if agent1_output is None or len(agent1_output.get("structured_profiles", [])) == 0:
        profiles = load_existing_structured_profiles(OUTPUT_DIR)
        if VERBOSE:
            print(f"[Agent1] Loaded {len(profiles)} existing structured profiles from disk.")
        agent1_output = {
            "agent": "agent1_collector_fallback",
            "target_count": len(COMPETITOR_TARGETS),
            "scrape_result_count": 0,
            "structured_profiles": profiles,
        }

    synthetic_generator = SyntheticEventGenerator(client)
    synthetic_events: dict[str, SyntheticCompetitorEvent] = {}

    enriched_profiles = []
    for profile in agent1_output["structured_profiles"]:
        sufficiency = profile.get("data_sufficiency", {})
        if not sufficiency.get("sufficient", True):
            event = synthetic_generator.generate(profile)
            synthetic_events[profile.get("company", "Unknown")] = event
            profile["synthetic_event"] = asdict(event)
            if VERBOSE:
                print(f"[Synthetic] Generated {event.event_type} event for {event.company}: {event.event_description}")
        enriched_profiles.append(profile)

    agent1_output["structured_profiles"] = enriched_profiles
    agent1_output["synthetic_event_count"] = len(synthetic_events)

    agent2_output = agent2.analyze(agent1_output)
    agent3_output = agent3.predict(agent1_output, agent2_output)
    agent4_output = agent4.recommend(agent2_output, agent3_output)

    elapsed = time.perf_counter() - start
    if VERBOSE:
        print(f"[Pipeline] Completed in {elapsed:.2f}s")

    return {
        "elapsed_seconds": elapsed,
        "agent1_output": agent1_output,
        "agent2_output": agent2_output,
        "agent3_output": agent3_output,
        "agent4_output": agent4_output,
    }

## 7. Run End-to-End Example (Agents 1-4)

In [16]:
pipeline_result = await run_agent_pipeline(run_live_scrape=RUN_LIVE_SCRAPE)

print("Agent 1 output keys:", list(pipeline_result["agent1_output"].keys()))
print("Agent 2 output keys:", list(pipeline_result["agent2_output"].keys()))
print("Agent 3 output keys:", list(pipeline_result["agent3_output"].keys()))
print("Agent 4 output keys:", list(pipeline_result["agent4_output"].keys()))

vulnerability_results = pipeline_result["agent2_output"]["vulnerability_results"]
pricing_predictions = pipeline_result["agent3_output"]["impact_forecasts"]
action_recommendations = pipeline_result["agent4_output"]["action_recommendations"]

print("Agent 2 report:", pipeline_result["agent2_output"]["report_path"])
print("Agent 3 report:", pipeline_result["agent3_output"]["report_path"])
print("Agent 4 report:", pipeline_result["agent4_output"]["report_path"])

[scrapers.crawlee_scraper] INFO  No APIFY_TOKEN found — running without proxy.  Set APIFY_TOKEN in .env for residential proxy rotation.
[crawlee.crawlers._playwright._playwright_crawler] INFO  Current request statistics:
┌───────────────────────────────┬──────┐
│ requests_finished             │ 0    │
│ requests_failed               │ 0    │
│ retry_histogram               │ [0]  │
│ request_avg_failed_duration   │ None │
│ request_avg_finished_duration │ None │
│ requests_finished_per_minute  │ 0    │
│ requests_failed_per_minute    │ 0    │
│ request_total_duration        │ 0s   │
│ requests_total                │ 0    │
│ crawler_runtime               │ 0s   │
└───────────────────────────────┴──────┘
[crawlee.crawlers._playwright._playwright_crawler] INFO  Crawled 0/5 pages, 0 failed requests, desired concurrency 3.


Using Groq inference (llama-3.3-70b-versatile)
[Agent1] Live scrape attempt 1/2


[crawlee._autoscaling.autoscaled_pool] INFO  current_concurrency = 0; desired_concurrency = 3; cpu = 0.0; mem = 0.0; event_loop = 0.0; client_info = 0.0
[crawlee._autoscaling.autoscaled_pool] INFO  Waiting for remaining tasks to finish
[crawlee.crawlers._playwright._playwright_crawler] INFO  Final request statistics:
┌───────────────────────────────┬─────────┐
│ requests_finished             │ 0       │
│ requests_failed               │ 0       │
│ retry_histogram               │ [0]     │
│ request_avg_failed_duration   │ None    │
│ request_avg_finished_duration │ None    │
│ requests_finished_per_minute  │ 0       │
│ requests_failed_per_minute    │ 0       │
│ request_total_duration        │ 0s      │
│ requests_total                │ 0       │
│ crawler_runtime               │ 448.1ms │
└───────────────────────────────┴─────────┘


[Agent1] Scraped 0 profiles.
[Agent1] Loaded 5 existing structured profiles from disk.
[Synthetic] Generated PRICE_CUT event for HubSpot: HubSpot has announced a significant price reduction for its Starter plan, aiming to attract more small businesses and solo entrepreneurs to its marketing, sales, and customer service platform. The new pricing strategy is set to disrupt the market, posing a challenge to competitors.
[Synthetic] Generated PRICE_CUT event for Stripe: Stripe announced a significant price cut for its online payment processing services, reducing the per-transaction fee for US-based merchants from $0.30 + 2.9% to $0.20 + 2.5% for transactions under $10.
[Agent2] Raw component names from LLM: ['pricing_pressure', 'segment_coverage', 'feature_depth', 'strategic_signals', 'business_model_pressure']
[Agent2] Raw component names from LLM: ['pricing_pressure', 'segment_coverage', 'feature_depth', 'strategic_signals', 'business_model_pressure']
[Agent2] Raw component names from LL

[openai._base_client] INFO  Retrying request to /chat/completions in 4.000000 seconds
[openai._base_client] INFO  Retrying request to /chat/completions in 5.000000 seconds
[openai._base_client] INFO  Retrying request to /chat/completions in 5.000000 seconds
[openai._base_client] INFO  Retrying request to /chat/completions in 5.000000 seconds
[openai._base_client] INFO  Retrying request to /chat/completions in 5.000000 seconds
[openai._base_client] INFO  Retrying request to /chat/completions in 5.000000 seconds


[Pipeline] Completed in 64.99s
Agent 1 output keys: ['agent', 'target_count', 'scrape_result_count', 'structured_profiles', 'synthetic_event_count']
Agent 2 output keys: ['agent', 'vulnerability_results', 'report_path', 'analyzed_count']
Agent 3 output keys: ['agent', 'impact_forecasts', 'report_path', 'forecast_count']
Agent 4 output keys: ['agent', 'action_recommendations', 'report_path', 'recommendation_count']
Agent 2 report: scraper_output/portfolio_risk_report.json
Agent 3 report: scraper_output/impact_forecast_report.json
Agent 4 report: scraper_output/action_recommendations_report.json


In [17]:
risk_rows = []
for result in vulnerability_results:
    top_component = max(result.component_breakdown, key=lambda c: c.weighted_score)
    risk_rows.append(
        {
            "company": result.company,
            "rank": result.peer_rank,
            "portfolio_risk_score": result.portfolio_risk_score,
            "risk_level": result.risk_level,
            "confidence": result.confidence,
            "top_driver": top_component.name,
            "signal_hits": sum(result.signals.values()),
            "is_synthetic": any(
                p.get("synthetic_event") is not None
                for p in pipeline_result["agent1_output"]["structured_profiles"]
                if p.get("company") == result.company
            ),
            "reasoning_summary": result.reasoning_summary,
        }
    )

pricing_rows = [
    {
        "company": p.company,
        "impact_probability": p.impact_probability,
        "pricing_risk": p.risk_level,
        "predicted_timeline": p.predicted_timeline,
        "top_driver": max(p.drivers.items(), key=lambda item: item[1])[0],
        "reasoning_summary": p.reasoning_summary,
    }
    for p in pricing_predictions
]

action_rows = [
    {
        "company": a.company,
        "priority": a.priority,
        "recommendation_type": a.recommendation_type,
        "owner": a.owner,
        "due_window": a.due_window,
        "action_title": a.action_title,
        "rationale": a.rationale,
    }
    for a in action_recommendations
]

risk_df = pd.DataFrame(risk_rows)
if not risk_df.empty:
    risk_df = risk_df.sort_values("rank")

pricing_df = pd.DataFrame(pricing_rows)
if not pricing_df.empty:
    pricing_df = pricing_df.sort_values("impact_probability", ascending=False)

actions_df = pd.DataFrame(action_rows)
if not actions_df.empty:
    actions_df = actions_df.sort_values("priority")

pd.set_option("display.max_colwidth", 220)
print("PORTFOLIO RISK ASSESSMENT")
display(risk_df)
print("IMPACT FORECAST")
display(pricing_df)
print("VC ACTION RECOMMENDATIONS")
display(actions_df)


PORTFOLIO RISK ASSESSMENT


,company,rank,portfolio_risk_score,risk_level,confidence,top_driver,signal_hits,is_synthetic,reasoning_summary
0,HubSpot,1,0.72,high,0.85,pricing_pressure,21,True,"Ranked #1/5 (peer_percentile=1.000). The synthetic event of HubSpot's significant price reduction for its Starter plan poses a substantial threat to the portfolio company's market position, particularly in terms of p..."
1,ClickUp,2,0.62,medium,0.85,pricing_pressure,21,False,"Ranked #2/5 (peer_percentile=0.750). ClickUp's aggressive pricing and feature expansion pose a moderate threat to RivalRadar Portfolio Co's market position, particularly in the mid-market segment. The competitor's AI..."
2,Linear,3,0.62,medium,0.85,pricing_pressure,21,False,"Ranked #3/5 (peer_percentile=0.500). Linear's competitive pricing and feature offerings pose a moderate threat to RivalRadar Portfolio Co's market position, particularly in terms of pricing pressure and feature parit..."
3,Notion,4,0.62,medium,0.85,pricing_pressure,21,False,"Ranked #4/5 (peer_percentile=0.250). Notion's competitive pricing and feature offerings pose a moderate threat to RivalRadar Portfolio Co's market position, particularly in the mid-market segment. The presence of a f..."
4,Stripe,5,0.62,medium,0.85,pricing_pressure,21,True,"Ranked #5/5 (peer_percentile=0.000). The competitor's recent price cut poses a moderate threat to the portfolio company's pricing power and customer retention, particularly in the mid-market segment. The price reduct..."


IMPACT FORECAST


,company,impact_probability,pricing_risk,predicted_timeline,top_driver,reasoning_summary
0,HubSpot,0.85,high,0-30 days,portfolio_competition_intensity,"The competitor, HubSpot, has announced a significant price reduction for its Starter plan, posing a substantial threat to the portfolio company's pricing power and customer retention. Given the price cut, customers m..."
1,Stripe,0.80,high,0-30 days,pricing_plan_volatility,"The competitor's synthetic event, a PRICE_CUT, poses a significant threat to the portfolio company's revenue and pricing power. The price cut announcement by Stripe may trigger a rapid response from customers, potent..."
2,ClickUp,0.75,medium,1-2 months,portfolio_competition_intensity,"The portfolio company is at risk due to ClickUp's strong integration strength and AI momentum, which could lead to feature parity and pricing pressure. The competitor's pricing summary shows a wide range of plans, in..."
3,Notion,0.75,medium,1-2 months,portfolio_competition_intensity,"The competitor Notion has a strong portfolio risk score and a medium risk level, indicating a potential threat to RivalRadar Portfolio Co's customer base. Notion's pricing structure and feature offerings could attrac..."
4,Linear,0.70,medium,1-2 months,analysis_confidence,"The competitor Linear poses a medium risk to RivalRadar Portfolio Co, primarily due to its pricing structure and feature offerings. The presence of a free tier and competitive pricing for the Basic and Business plans..."


VC ACTION RECOMMENDATIONS


,company,priority,recommendation_type,owner,due_window,action_title,rationale
0,ClickUp,P1,gtm_response,Portfolio Operations Lead,within 2 weeks,Competitive Threat Assessment,"This assessment will help the VC firm understand the potential risks and opportunities arising from ClickUp's competitive moves, enabling proactive engagement with RivalRadar Portfolio Co to mitigate potential threat..."
1,HubSpot,P1,gtm_response,Portfolio Operations Lead,within 2 weeks,Competitive Pricing Review,This review will help protect the investment value by ensuring the portfolio company is prepared to respond to potential competitive pressure and maintain its market position.
2,Linear,P1,gtm_response,Portfolio Operations Lead,within 2 weeks,Competitor Pricing Review,This review will help protect the investment value by ensuring RivalRadar Portfolio Co remains competitive in the market and can respond effectively to pricing pressure from Linear.
3,Notion,P1,gtm_response,Portfolio Operations Lead,within 2 weeks,Competitor Pricing Review,"Protecting investment value by proactively assessing and responding to competitive pricing pressure, ensuring RivalRadar Portfolio Co remains competitive in the mid-market segment."
4,Stripe,P1,gtm_response,Portfolio Operations Lead,within 2 weeks,Review Pricing Strategy,This recommendation protects investment value by ensuring the portfolio company is prepared to respond to the competitor's price cut and maintain its pricing power and customer retention.


## 8. Validation and Debug Output (Agents 1-4)

In [18]:
agent1_output = pipeline_result["agent1_output"]
agent2_output = pipeline_result["agent2_output"]
agent3_output = pipeline_result["agent3_output"]
agent4_output = pipeline_result["agent4_output"]

assert "structured_profiles" in agent1_output, "Agent 1 output missing structured_profiles"
assert isinstance(agent1_output["structured_profiles"], list), "structured_profiles must be a list"
assert len(agent1_output["structured_profiles"]) > 0, "No structured profiles available"

assert "vulnerability_results" in agent2_output, "Agent 2 output missing vulnerability_results"
assert isinstance(agent2_output["vulnerability_results"], list), "vulnerability_results must be a list"
assert len(agent2_output["vulnerability_results"]) > 0, "Agent 2 returned no vulnerability results"

assert "impact_forecasts" in agent3_output, "Agent 3 output missing impact_forecasts"
assert isinstance(agent3_output["impact_forecasts"], list), "impact_forecasts must be a list"
assert len(agent3_output["impact_forecasts"]) > 0, "Agent 3 returned no impact forecasts"

assert "action_recommendations" in agent4_output, "Agent 4 output missing action_recommendations"
assert isinstance(agent4_output["action_recommendations"], list), "action_recommendations must be a list"
assert len(agent4_output["action_recommendations"]) > 0, "Agent 4 returned no recommendations"

sample_profile = agent1_output["structured_profiles"][0]
required_profile_keys = {"company", "pricing_summary", "raw_char_count"}
assert required_profile_keys.issubset(sample_profile.keys()), "Profile schema mismatch"

if VERBOSE:
    print("[Validation] Agents 1-4 outputs passed schema checks.")
    print("[Debug] Structured profiles:", len(agent1_output["structured_profiles"]))
    print("[Debug] Vulnerability results:", len(agent2_output["vulnerability_results"]))
    print("[Debug] Impact forecasts:", len(agent3_output["impact_forecasts"]))
    print("[Debug] Action recommendations:", len(agent4_output["action_recommendations"]))
    print("[Debug] Pipeline elapsed seconds:", round(pipeline_result["elapsed_seconds"], 2))

[Validation] Agents 1-4 outputs passed schema checks.
[Debug] Structured profiles: 5
[Debug] Vulnerability results: 5
[Debug] Impact forecasts: 5
[Debug] Action recommendations: 5
[Debug] Pipeline elapsed seconds: 64.99
